# BankChurners Portable Retest

This notebook reruns the `billing2` portable model on BankChurners to check cross-domain generality.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from xgboost import XGBClassifier


ROOT = Path("/home/ms990728/소정해")
RESULT_PATH = ROOT / "results" / "bankchurners_portable_retest_results.md"
NOTEBOOK_PATH = ROOT / "notebooks" / "comparison" / "bankchurners_portable_retest.ipynb"
RANDOM_STATE = 42

XGB_PARAMS = {
    "colsample_bytree": 0.8334271527847554,
    "gamma": 0.11429345433755263,
    "learning_rate": 0.06857996256539675,
    "max_depth": 3,
    "min_child_weight": 2,
    "n_estimators": 493,
    "reg_alpha": 0.2497327922401265,
    "reg_lambda": 0.9246782213565523,
    "subsample": 0.7954562418017752,
}


def load_cell2cell(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=["NA", ""], keep_default_na=True)
    df["HandsetPrice"] = pd.to_numeric(df["HandsetPrice"].replace("Unknown", pd.NA), errors="coerce")
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
    return df


def load_bankchurners(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Attrition_Flag"] = df["Attrition_Flag"].map({"Existing Customer": 0, "Attrited Customer": 1}).astype(int)
    return df


def build_billing2_cell(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X = pd.DataFrame(
        {
            "monthly_billing": pd.to_numeric(df["MonthlyRevenue"], errors="coerce"),
            "total_billing": pd.to_numeric(df["TotalRecurringCharge"], errors="coerce"),
        }
    )
    y = df["Churn"].astype(int)
    return X, y


def build_billing2_bank(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X = pd.DataFrame(
        {
            "monthly_billing": pd.to_numeric(df["Total_Trans_Amt"], errors="coerce"),
            "total_billing": pd.to_numeric(df["Credit_Limit"], errors="coerce"),
        }
    )
    y = df["Attrition_Flag"].astype(int)
    return X, y


def split_domain(X: pd.DataFrame, y: pd.Series):
    return train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )


def fit_imputer(X_train: pd.DataFrame):
    imputer = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    return imputer, X_train_imp


def transform_imputer(imputer: SimpleImputer, X: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(imputer.transform(X), columns=X.columns, index=X.index)


def fit_rank_transform(X_train: pd.DataFrame):
    qt = QuantileTransformer(
        output_distribution="uniform",
        random_state=RANDOM_STATE,
        n_quantiles=min(len(X_train), 1000),
    )
    X_train_rank = pd.DataFrame(qt.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    return qt, X_train_rank


def transform_rank(qt: QuantileTransformer, X: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(qt.transform(X), columns=X.columns, index=X.index)


def _sqrtm_psd(mat: np.ndarray, inverse: bool = False, eps: float = 1e-6) -> np.ndarray:
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.maximum(eigvals, eps)
    if inverse:
        values = 1.0 / np.sqrt(eigvals)
    else:
        values = np.sqrt(eigvals)
    return eigvecs @ np.diag(values) @ eigvecs.T


def fit_coral(source_train: pd.DataFrame, target_train: pd.DataFrame) -> dict:
    xs = np.asarray(source_train, dtype=float)
    xt = np.asarray(target_train, dtype=float)
    mu_s = xs.mean(axis=0, keepdims=True)
    mu_t = xt.mean(axis=0, keepdims=True)
    cs = np.cov(xs - mu_s, rowvar=False) + 1e-6 * np.eye(xs.shape[1])
    ct = np.cov(xt - mu_t, rowvar=False) + 1e-6 * np.eye(xt.shape[1])
    a = _sqrtm_psd(cs, inverse=True) @ _sqrtm_psd(ct, inverse=False)
    return {"mu_s": mu_s, "mu_t": mu_t, "a": a}


def transform_coral(X: pd.DataFrame, params: dict) -> pd.DataFrame:
    xs = np.asarray(X, dtype=float)
    aligned = (xs - params["mu_s"]) @ params["a"] + params["mu_t"]
    return pd.DataFrame(aligned, columns=X.columns, index=X.index)


def fit_xgb(X_train: pd.DataFrame, y_train: pd.Series) -> XGBClassifier:
    scale_pos_weight = int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1)
    model = XGBClassifier(
        **XGB_PARAMS,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def fit_domain_classifier(X_source: pd.DataFrame, X_target: pd.DataFrame) -> dict:
    X = pd.concat([X_source, X_target], axis=0, ignore_index=True)
    y = np.array([0] * len(X_source) + [1] * len(X_target))
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    model = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "domain_auc": roc_auc_score(y_test, prob),
        "domain_accuracy": accuracy_score(y_test, pred),
    }


def compute_metrics(y_true: pd.Series, prob: np.ndarray, threshold: float = 0.5) -> dict:
    pred = (prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def best_f1_threshold(y_true: pd.Series, prob: np.ndarray) -> tuple[float, float, dict]:
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = np.array([f1_score(y_true, (prob >= t).astype(int), zero_division=0) for t in thresholds])
    idx = int(scores.argmax())
    threshold = float(thresholds[idx])
    return threshold, float(scores[idx]), compute_metrics(y_true, prob, threshold=threshold)


def md_table(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |\n"
    header += "|" + "|".join(["---"] * len(cols)) + "|\n"
    rows = []
    for _, row in df.iterrows():
        rows.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return header + "\n".join(rows)


def evaluate_raw(src_train, src_test, src_y_train, src_y_test, tgt_train, tgt_test, tgt_y_train, tgt_y_test):
    src_imp, src_train_imp = fit_imputer(src_train)
    tgt_imp, tgt_train_imp = fit_imputer(tgt_train)
    src_test_imp = transform_imputer(src_imp, src_test)
    tgt_test_imp = transform_imputer(tgt_imp, tgt_test)

    src_model = fit_xgb(src_train_imp, src_y_train)
    tgt_model = fit_xgb(tgt_train_imp, tgt_y_train)

    src_holdout_prob = src_model.predict_proba(src_test_imp)[:, 1]
    tgt_holdout_prob = tgt_model.predict_proba(tgt_test_imp)[:, 1]
    c2b_prob = src_model.predict_proba(tgt_test_imp)[:, 1]
    b2c_prob = tgt_model.predict_proba(src_test_imp)[:, 1]

    domain_metrics = fit_domain_classifier(src_train_imp, tgt_train_imp)

    return {
        "src_holdout": compute_metrics(src_y_test, src_holdout_prob),
        "tgt_holdout": compute_metrics(tgt_y_test, tgt_holdout_prob),
        "c2b": compute_metrics(tgt_y_test, c2b_prob),
        "b2c": compute_metrics(src_y_test, b2c_prob),
        "c2b_best": best_f1_threshold(tgt_y_test, c2b_prob),
        "b2c_best": best_f1_threshold(src_y_test, b2c_prob),
        "domain": domain_metrics,
    }


def evaluate_rank(src_train, src_test, src_y_train, src_y_test, tgt_train, tgt_test, tgt_y_train, tgt_y_test):
    src_imp, src_train_imp = fit_imputer(src_train)
    tgt_imp, tgt_train_imp = fit_imputer(tgt_train)
    src_test_imp = transform_imputer(src_imp, src_test)
    tgt_test_imp = transform_imputer(tgt_imp, tgt_test)

    src_qt, src_train_rank = fit_rank_transform(src_train_imp)
    tgt_qt, tgt_train_rank = fit_rank_transform(tgt_train_imp)
    src_test_rank = transform_rank(src_qt, src_test_imp)
    tgt_test_rank = transform_rank(tgt_qt, tgt_test_imp)

    src_model = fit_xgb(src_train_rank, src_y_train)
    tgt_model = fit_xgb(tgt_train_rank, tgt_y_train)

    src_holdout_prob = src_model.predict_proba(src_test_rank)[:, 1]
    tgt_holdout_prob = tgt_model.predict_proba(tgt_test_rank)[:, 1]
    c2b_prob = src_model.predict_proba(tgt_test_rank)[:, 1]
    b2c_prob = tgt_model.predict_proba(src_test_rank)[:, 1]

    domain_metrics = fit_domain_classifier(src_train_rank, tgt_train_rank)

    return {
        "src_holdout": compute_metrics(src_y_test, src_holdout_prob),
        "tgt_holdout": compute_metrics(tgt_y_test, tgt_holdout_prob),
        "c2b": compute_metrics(tgt_y_test, c2b_prob),
        "b2c": compute_metrics(src_y_test, b2c_prob),
        "c2b_best": best_f1_threshold(tgt_y_test, c2b_prob),
        "b2c_best": best_f1_threshold(src_y_test, b2c_prob),
        "domain": domain_metrics,
    }


def evaluate_coral(src_train, src_test, src_y_train, src_y_test, tgt_train, tgt_test, tgt_y_train, tgt_y_test):
    src_imp, src_train_imp = fit_imputer(src_train)
    tgt_imp, tgt_train_imp = fit_imputer(tgt_train)
    src_test_imp = transform_imputer(src_imp, src_test)
    tgt_test_imp = transform_imputer(tgt_imp, tgt_test)

    src_to_tgt = fit_coral(src_train_imp, tgt_train_imp)
    tgt_to_src = fit_coral(tgt_train_imp, src_train_imp)

    src_train_coral = transform_coral(src_train_imp, src_to_tgt)
    src_test_coral = transform_coral(src_test_imp, src_to_tgt)
    tgt_train_coral = transform_coral(tgt_train_imp, tgt_to_src)
    tgt_test_coral = transform_coral(tgt_test_imp, tgt_to_src)

    src_model = fit_xgb(src_train_coral, src_y_train)
    tgt_model = fit_xgb(tgt_train_coral, tgt_y_train)

    src_holdout_prob = src_model.predict_proba(src_test_coral)[:, 1]
    tgt_holdout_prob = tgt_model.predict_proba(tgt_test_coral)[:, 1]
    c2b_prob = src_model.predict_proba(tgt_test_imp)[:, 1]
    b2c_prob = tgt_model.predict_proba(src_test_imp)[:, 1]

    domain_metrics = fit_domain_classifier(src_train_coral, tgt_train_imp)

    return {
        "src_holdout": compute_metrics(src_y_test, src_holdout_prob),
        "tgt_holdout": compute_metrics(tgt_y_test, tgt_holdout_prob),
        "c2b": compute_metrics(tgt_y_test, c2b_prob),
        "b2c": compute_metrics(src_y_test, b2c_prob),
        "c2b_best": best_f1_threshold(tgt_y_test, c2b_prob),
        "b2c_best": best_f1_threshold(src_y_test, b2c_prob),
        "domain": domain_metrics,
    }


def summarize_method(name: str, res: dict) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    holdout = pd.DataFrame(
        [
            {"split": "Cell2Cell holdout", "method": name, **res["src_holdout"]},
            {"split": "BankChurners holdout", "method": name, **res["tgt_holdout"]},
        ]
    )
    transfer = pd.DataFrame(
        [
            {"direction": "Cell2Cell -> BankChurners", "method": name, **res["c2b"], "inverted_auc": 1 - res["c2b"]["roc_auc"]},
            {"direction": "BankChurners -> Cell2Cell", "method": name, **res["b2c"], "inverted_auc": 1 - res["b2c"]["roc_auc"]},
        ]
    )
    threshold = pd.DataFrame(
        [
            {
                "direction": "Cell2Cell -> BankChurners",
                "method": name,
                "best_threshold": res["c2b_best"][0],
                "best_f1": res["c2b_best"][1],
                **res["c2b_best"][2],
            },
            {
                "direction": "BankChurners -> Cell2Cell",
                "method": name,
                "best_threshold": res["b2c_best"][0],
                "best_f1": res["b2c_best"][1],
                **res["b2c_best"][2],
            },
        ]
    )
    return holdout, transfer, threshold


def build_result_text(source_df, bank_df, holdout_df, transfer_df, threshold_df, domain_df):
    result_text = []
    result_text.append("# BankChurners Portable Retest")
    result_text.append("")
    result_text.append("## Setup")
    result_text.append("- Source domain: Cell2Cell")
    result_text.append("- Target domain: BankChurners")
    result_text.append("- Portable schema: `billing2 = monthly_billing + total_billing`")
    result_text.append("- Cell2Cell mapping: `MonthlyRevenue`, `TotalRecurringCharge`")
    result_text.append("- BankChurners mapping: `Total_Trans_Amt`, `Credit_Limit`")
    result_text.append("- Methods: `raw`, `rank normalization`, `CORAL`")
    result_text.append("")
    result_text.append("## Dataset Size")
    result_text.append(f"- Cell2Cell rows: `{len(source_df):,}`; churn rate `{source_df['Churn'].mean():.4f}`")
    result_text.append(f"- BankChurners rows: `{len(bank_df):,}`; attrition rate `{bank_df['Attrition_Flag'].mean():.4f}`")
    result_text.append("")
    result_text.append("## Cell2Cell Holdout")
    result_text.append(md_table(holdout_df[holdout_df["split"] == "Cell2Cell holdout"].round(4)))
    result_text.append("")
    result_text.append("## BankChurners Holdout")
    result_text.append(md_table(holdout_df[holdout_df["split"] == "BankChurners holdout"].round(4)))
    result_text.append("")
    result_text.append("## Cell2Cell -> BankChurners Transfer")
    result_text.append(md_table(transfer_df[transfer_df["direction"] == "Cell2Cell -> BankChurners"].round(4)))
    result_text.append("")
    result_text.append("## BankChurners -> Cell2Cell Transfer")
    result_text.append(md_table(transfer_df[transfer_df["direction"] == "BankChurners -> Cell2Cell"].round(4)))
    result_text.append("")
    result_text.append("## Cell2Cell -> BankChurners Threshold Sensitivity")
    result_text.append(md_table(threshold_df[threshold_df["direction"] == "Cell2Cell -> BankChurners"].round(4)))
    result_text.append("")
    result_text.append("## BankChurners -> Cell2Cell Threshold Sensitivity")
    result_text.append(md_table(threshold_df[threshold_df["direction"] == "BankChurners -> Cell2Cell"].round(4)))
    result_text.append("")
    result_text.append("## Domain Separability")
    result_text.append(md_table(domain_df.round(4)))
    result_text.append("")
    result_text.append("## Interpretation")
    result_text.append("- `billing2` is still too small to carry over cleanly to BankChurners.")
    result_text.append("- If CORAL helps only weakly, the problem is not just scale mismatch; the feature meaning itself is drifting.")
    result_text.append("- If AUC stays below 0.5, the portable representation is not yet truly general across domains.")
    result_text.append("- This bank retest therefore checks portability, but it does not yet confirm a fully general model.")
    return "\n".join(result_text)


def write_notebook(script_text: str):
    nb = {
        "cells": [
            {
                "cell_type": "markdown",
                "metadata": {},
                "source": [
                    "# BankChurners Portable Retest\n",
                    "\n",
                    "This notebook reruns the `billing2` portable model on BankChurners to check cross-domain generality.\n",
                ],
            },
            {
                "cell_type": "code",
                "execution_count": None,
                "metadata": {},
                "outputs": [],
                "source": script_text.splitlines(keepends=True),
            },
        ],
        "metadata": {
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3",
            },
            "language_info": {
                "name": "python",
                "version": "3.11",
            },
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    NOTEBOOK_PATH.write_text(json.dumps(nb, ensure_ascii=False, indent=2), encoding="utf-8")


def main():
    cell_df = load_cell2cell(ROOT / "data" / "raw" / "cell2celltrain.csv")
    bank_df = load_bankchurners(ROOT / "data" / "external" / "credit_card_churn" / "BankChurners.csv")

    cell_X, cell_y = build_billing2_cell(cell_df)
    bank_X, bank_y = build_billing2_bank(bank_df)

    c_X_train, c_X_test, c_y_train, c_y_test = split_domain(cell_X, cell_y)
    b_X_train, b_X_test, b_y_train, b_y_test = split_domain(bank_X, bank_y)

    raw_res = evaluate_raw(c_X_train, c_X_test, c_y_train, c_y_test, b_X_train, b_X_test, b_y_train, b_y_test)
    rank_res = evaluate_rank(c_X_train, c_X_test, c_y_train, c_y_test, b_X_train, b_X_test, b_y_train, b_y_test)
    coral_res = evaluate_coral(c_X_train, c_X_test, c_y_train, c_y_test, b_X_train, b_X_test, b_y_train, b_y_test)

    tables = {
        "raw": summarize_method("raw", raw_res),
        "rank": summarize_method("rank", rank_res),
        "coral": summarize_method("coral", coral_res),
    }

    holdout_df = pd.concat([tables["raw"][0], tables["rank"][0], tables["coral"][0]], ignore_index=True)
    transfer_df = pd.concat([tables["raw"][1], tables["rank"][1], tables["coral"][1]], ignore_index=True)
    threshold_df = pd.concat([tables["raw"][2], tables["rank"][2], tables["coral"][2]], ignore_index=True)
    domain_df = pd.DataFrame(
        [
            {"method": "raw", **raw_res["domain"]},
            {"method": "rank", **rank_res["domain"]},
            {"method": "coral", **coral_res["domain"]},
        ]
    )

    result_text = build_result_text(cell_df, bank_df, holdout_df, transfer_df, threshold_df, domain_df)
    RESULT_PATH.write_text(result_text, encoding="utf-8")

    print("Wrote:", RESULT_PATH)
    print()
    print("=== Cell2Cell holdout ===")
    print(md_table(holdout_df[holdout_df["split"] == "Cell2Cell holdout"].round(4)))
    print()
    print("=== BankChurners holdout ===")
    print(md_table(holdout_df[holdout_df["split"] == "BankChurners holdout"].round(4)))
    print()
    print("=== Cell2Cell -> BankChurners transfer ===")
    print(md_table(transfer_df[transfer_df["direction"] == "Cell2Cell -> BankChurners"].round(4)))
    print()
    print("=== BankChurners -> Cell2Cell transfer ===")
    print(md_table(transfer_df[transfer_df["direction"] == "BankChurners -> Cell2Cell"].round(4)))

    script_path = Path(__file__) if "__file__" in globals() else None
    if script_path is not None and script_path.exists():
        write_notebook(script_path.read_text(encoding="utf-8"))


if __name__ == "__main__":
    main()
